# AMEX Enterprise Credit Risk Platform
## Notebook 34 -- Early Payment Default: Business Understanding & Policy
### Phase 2 . Problem Statement 5: Early Payment Default Detection

CRISP-DM stage: **Business Understanding**. Sprint 1, Notebook 1 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 (reads `project_config.json`, `notebook_02_summary.json`, `notebook_05_summary.json`) -- no dependency on Problem 3 or Problem 4.

**What this notebook does (real, computed on your machine when you run it):**
- States the business case for early risk detection and an honest, explicit data-limitation statement (this dataset has no account-origination date)
- Reads the REAL per-customer statement-count distribution directly from the raw Kaggle training CSV (not from Problem 1's already-aggregated feature files, which carry only one row per customer)
- Defines `EARLY_WINDOW_CANDIDATES` -- a set of genuine calendar-quarter checkpoints (3/6/9/12 months, each strictly below the dataset's real measured statement-count ceiling) to test for "early" prediction, replacing an earlier single-percentile-derived K that this notebook found collapsed to the ceiling itself for 84% of customers (see Section 5's "IMPORTANT FINDING")
- Sets an explicit, labeled `ASSUMPTION` KPI target (>=80% of Notebook 05's full-history AUC retained) that Notebook 35 (Modeling) will be validated against
- Writes `early_default_policy.json` for Notebook 35 to consume

**What this notebook does NOT do:** no modeling, no restricted-window feature engineering yet -- that's Notebook 35. This notebook only establishes the policy and the real data facts that policy depends on.

Zero-fabrication: every number this notebook prints is computed live from your real Kaggle data on this run. `ASSUMPTION`-labeled values (the K floor, the KPI target) are explicit, editable business choices, not disguised as measured facts.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Problem 1 (Notebooks 01-05)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(
            f"{_p} not found.\nFix: {_fix} -- Problem 5 depends on Problem 1's real "
            f"champion model and data engineering outputs, not on Problem 3 or 4."
        )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

# Problem 5's own output directory -- a NEW pillar this notebook introduces
# (Phase 2, Problem 5). If project_config.json's pillar_dirs already has an
# "early_payment_default_policy" entry, that path is used; otherwise this
# falls back to the platform's standard Phase/Problem folder convention and
# creates it -- the same defensive-fallback idiom every notebook in this
# platform already uses for an older config missing resource_limits.
if "early_payment_default_policy" in PILLAR_DIRS:
    EPD_POLICY_DIR = PILLAR_DIRS["early_payment_default_policy"]
else:
    EPD_POLICY_DIR = (
        PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning"
        / "05_Problem5_Early_Payment_Default_Detection" / "policy"
    )
    print(
        "NOTE: 'early_payment_default_policy' not found in project_config.json's "
        "pillar_dirs -- using the standard folder-convention fallback:\n"
        f"      {EPD_POLICY_DIR}\n"
        "      (If you've registered a different path for this pillar, edit the "
        "PILLAR_DIRS lookup above to match it.)"
    )
EPD_POLICY_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]

print(f"Loaded config from      : {CONFIG_PATH}")
print(f"RANDOM_SEED              : {RANDOM_SEED} (same seed used by every notebook in this platform)")
print(f"WARP_THREAD_COUNT        : {WARP_THREAD_COUNT}")
print(f"Champion model (Problem 1, measured)    : {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured)         : {CHAMPION_METRICS.get('holdout_auc')}")
print(f"Champion holdout AMEX metric (measured) : {CHAMPION_METRICS.get('holdout_amex_metric')}")
print(f"Policy artifacts will be written under  : {EPD_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    """Current process resident memory, in GB."""
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: BUSINESS UNDERSTANDING -- WHY EARLY DETECTION MATTERS
# =============================================================================
_section("SECTION 3: Business Understanding -- Why Early Detection Matters")

print(
    "Every month a high-risk customer goes undetected is a month of lost "
    "intervention options: a credit-line reduction, a payment-plan offer, an "
    "outreach call, or a reserve adjustment all work better -- and cost less -- "
    "the earlier they happen relative to when a customer's risk actually starts "
    "rising. Problem 1's champion model (Notebook 05) already predicts default "
    "using a customer's FULL available statement history. This problem asks a "
    "narrower, more operationally relevant question: how much of that same "
    "predictive power survives if only the customer's EARLIEST statements are "
    "available -- the situation a real early-warning system is actually in."
)
print(
    "\nDATA LIMITATION (stated plainly, same standard as Problem 2's fair-lending "
    "section): this Kaggle dataset carries no account-origination date, so "
    "'Early Payment Default' cannot be computed in its traditional lending sense "
    "(default within N months of account opening). What IS real and computable "
    "in this data is each customer's OWN earliest-available statements -- so "
    "this notebook reframes the problem honestly as: given only a customer's "
    "first K monthly statements, how well can eventual default still be "
    "predicted? -- and documents that reframing here rather than silently "
    "presenting it as literal time-since-origination."
)
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION
# =============================================================================
_section("SECTION 4: Real Per-Customer Statement-Count Distribution (Determines the 'Early Window' K)")

# CORRECTION (found on a real run): train_full_features.parquet is NOT raw
# per-statement data -- a real run showed every customer with exactly 1 row
# (min = p10 = p25 = median = mean = max = 1 statement/customer), proving
# it's already aggregated to one row per customer_ID by Notebook 02, same
# shape as Notebook 04's "engineered" output, just an earlier stage of it.
# The real per-statement data (multiple rows per customer_ID, one per
# monthly S_2 date) only exists in the raw Kaggle CSV itself, which this
# platform's own README documents as not redistributed inside the project
# folder (see data/README.md) -- it lives in the separate raw-data folder
# set up before Notebook 01 was first run.
_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break

if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv (needed for real per-statement "
        "row counts -- train_full_features.parquet is already aggregated to one "
        "row per customer, confirmed on a real run, so it can't be used here). "
        "Checked:\n"
        + "\n".join(f"  - {c}" for c in _raw_candidates)
        + "\n\nFor diagnosis, here is what's actually available:\n"
        f"  PROJECT_CONFIG top-level keys: {sorted(PROJECT_CONFIG.keys())}\n"
        f"  NB02_SUMMARY['output_files'] keys: {sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: tell me the real path to your raw train_data.csv so this can be "
        "corrected with the real path rather than another guess."
    )

print(f"Reading real per-statement (raw, pre-aggregation) data from: {RAW_TRAIN_DATA_PATH}")
print("(This scans the full raw CSV with only the customer_ID column projected "
      "-- may take a minute or two on a 16GB+ file.)")
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
STATEMENT_COUNT_STATS = {
    "n_customers": _n_customers,
    "min": int(_counts_series.min()),
    "p10": float(_counts_series.quantile(0.10)),
    "p25": float(_counts_series.quantile(0.25)),
    "median": float(_counts_series.median()),
    "mean": float(_counts_series.mean()),
    "max": int(_counts_series.max()),
}
print(f"Customers (real, measured): {STATEMENT_COUNT_STATS['n_customers']:,}")
for _k in ("min", "p10", "p25", "median", "mean", "max"):
    print(f"  {_k:>6} statements/customer: {STATEMENT_COUNT_STATS[_k]}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: EARLY-WINDOW POLICY -- DEFINING K (ASSUMPTION)
# =============================================================================
_section("SECTION 5: Early-Window Policy -- Defining Candidate Windows (ASSUMPTION)")

print(
    "IMPORTANT FINDING (real, measured): this dataset's statement history is "
    f"right-censored at {STATEMENT_COUNT_STATS['max']} months (the real max "
    "statements/customer), and most customers sit close to that ceiling "
    f"(median={STATEMENT_COUNT_STATS['median']:.0f}, "
    f"mean={STATEMENT_COUNT_STATS['mean']:.2f}). A single percentile-derived K "
    "(e.g. the 25th percentile, which computed to the ceiling value itself on "
    "this real data) would make 'first K statements' equal to FULL history for "
    "most customers -- trivially comparing the full-history model to itself "
    "rather than testing genuine early detection. Dropping that approach for "
    "this reason rather than using a K that would silently defeat the point "
    "of this problem."
)

# ASSUMPTION: instead of one percentile-derived K, test a genuine range of
# calendar-quarter checkpoints (3/6/9/12 months) -- each meaningfully SMALLER
# than the dataset's real ceiling, so each is an actual test of "how early is
# early enough", not a disguised full-history re-run. Only candidates
# strictly below the real measured max are kept (defensive, in case a
# different data vintage has a different observation-window length).
_candidate_windows = [3, 6, 9, 12]
EARLY_WINDOW_CANDIDATES = sorted({k for k in _candidate_windows if k < STATEMENT_COUNT_STATS["max"]})
if not EARLY_WINDOW_CANDIDATES:
    # Degenerate fallback (should not occur with this dataset's real 13-month
    # ceiling) -- guarantees at least one candidate strictly below the ceiling.
    EARLY_WINDOW_CANDIDATES = [max(3, STATEMENT_COUNT_STATS["max"] - 1)]

EARLY_WINDOW_COVERAGE = {
    k: float((_counts_series >= k).sum() / _n_customers * 100.0)
    for k in EARLY_WINDOW_CANDIDATES
}

print(f"\nEARLY_WINDOW_CANDIDATES (ASSUMPTION -- quarterly checkpoints below "
      f"the real ceiling of {STATEMENT_COUNT_STATS['max']}): {EARLY_WINDOW_CANDIDATES}")
for _k, _pct in EARLY_WINDOW_COVERAGE.items():
    print(f"  K={_k:>2}: {_pct:.1f}% of customers have >= {_k} statements available (measured)")

print(
    "\nNotebook 35 (Modeling) will, for EACH candidate K above, re-aggregate "
    "each customer's first K chronologically-earliest statements only (by "
    "real S_2 date order, not by row order) using the same feature-engineering "
    "transformations Notebook 04 already established, then train and evaluate "
    "a model against that early-only feature set -- reporting an AUC-retention "
    "curve across window lengths (Section 6's KPI target applies at each K), "
    "rather than a single yes/no answer at one arbitrary window length."
)
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SUCCESS CRITERIA (ASSUMPTION)
# =============================================================================
_section("SECTION 6: Success Criteria (ASSUMPTION)")

EPD_KPI_TARGETS = {
    "min_auc_retention_vs_full_history": 0.80,
    "description": (
        "ASSUMPTION -- the early-window model (Notebook 35) should retain at "
        "least 80% of Notebook 05's full-history champion holdout AUC (i.e. "
        "early_auc / full_history_auc >= 0.80) to be considered operationally "
        "useful for early risk flagging. Below that threshold, the honest "
        "conclusion is 'not enough signal yet at this window length' -- Notebook "
        "35 must report that outcome plainly if it happens, not obscure it."
    ),
    "full_history_reference_auc": CHAMPION_METRICS.get("holdout_auc"),
}
print(json.dumps(EPD_KPI_TARGETS, indent=2))
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: TARGET DEFINITION -- UNCHANGED FROM PROBLEM 1
# =============================================================================
_section("SECTION 7: Target Definition")

print(
    "This problem does NOT redefine the outcome being predicted -- it reuses "
    "the exact same eventual-default target Problem 1 uses (from the real "
    "train_labels.csv). Only the FEATURES available to the model differ "
    "(first-K-statements-only vs. full history). This keeps the AUC comparison "
    "against Notebook 05 in Section 6 a valid apples-to-apples-but-with-less-"
    "information benchmark, rather than comparing two different questions."
)
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: WRITE EARLY-DEFAULT POLICY ARTIFACT
# =============================================================================
_section("SECTION 8: Write Early-Default Policy Artifact")

EARLY_DEFAULT_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 5 -- Early Payment Default Detection",
    "data_limitation_statement": (
        "This dataset has no account-origination date. 'Early' is defined as "
        "each customer's earliest N available monthly statements (by real "
        "chronological S_2 order), not time-since-account-opening, tested "
        "across the candidate window lengths in early_window_candidates below "
        "rather than a single fixed N. See this notebook's Section 3 and "
        "Section 5 for the full reasoning, including why a single "
        "percentile-derived N was rejected."
    ),
    "early_window_candidates": EARLY_WINDOW_CANDIDATES,
    "early_window_coverage_by_k": EARLY_WINDOW_COVERAGE,
    "statement_count_stats": STATEMENT_COUNT_STATS,
    "target_definition": (
        "Same eventual-default target as Problem 1 (train_labels.csv) -- only "
        "the FEATURES available differ (first K statements vs. full history), "
        "not the outcome being predicted."
    ),
    "kpi_targets": EPD_KPI_TARGETS,
    "champion_model_reference": CHAMPION_NAME,
    "champion_holdout_auc_reference": CHAMPION_METRICS.get("holdout_auc"),
    "champion_holdout_amex_metric_reference": CHAMPION_METRICS.get("holdout_amex_metric"),
    "random_seed": RANDOM_SEED,
}

POLICY_PATH = EPD_POLICY_DIR / "early_default_policy.json"
with open(POLICY_PATH, "w", encoding="utf-8") as f:
    json.dump(EARLY_DEFAULT_POLICY, f, indent=2)
print(f"Wrote: {POLICY_PATH}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 9: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", POLICY_PATH.exists())
_all_checks_passed &= _check(
    "EARLY_WINDOW_CANDIDATES is a non-empty list, all values >= 3",
    len(EARLY_WINDOW_CANDIDATES) > 0 and all(k >= 3 for k in EARLY_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Every candidate window is strictly below the real measured ceiling",
    all(k < STATEMENT_COUNT_STATS["max"] for k in EARLY_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Early-window coverage is a real measured percentage in (0, 100] for every candidate",
    all(0.0 < pct <= 100.0 for pct in EARLY_WINDOW_COVERAGE.values()),
)
_all_checks_passed &= _check(
    "Statement count stats are internally consistent (min <= p25 <= max)",
    STATEMENT_COUNT_STATS["min"] <= STATEMENT_COUNT_STATS["p25"] <= STATEMENT_COUNT_STATS["max"],
)
_all_checks_passed &= _check(
    "Reused Problem 1's real champion AUC (not fabricated)",
    EPD_KPI_TARGETS["full_history_reference_auc"] == CHAMPION_METRICS.get("holdout_auc"),
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 9 complete -- all checks passed.")


# =============================================================================
# SECTION 10: WRITE NOTEBOOK 34 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 10: Write Notebook 34 Summary Artifact")

NB34_SUMMARY = {
    "notebook": "34_early_payment_default_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "early_window_candidates": EARLY_WINDOW_CANDIDATES,
    "early_window_coverage_by_k": EARLY_WINDOW_COVERAGE,
    "policy_path": str(POLICY_PATH),
    "kpi_targets": EPD_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}
NB34_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_34_summary.json"
with open(NB34_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB34_SUMMARY, f, indent=2)
print(f"Wrote: {NB34_SUMMARY_PATH}")

_section("NOTEBOOK 34 COMPLETE")
print(f"Early-window candidates (real, computed): {EARLY_WINDOW_CANDIDATES}")
for _k, _pct in EARLY_WINDOW_COVERAGE.items():
    print(f"  K={_k:>2}: {_pct:.1f}% of customers covered")
print(f"Policy written to: {POLICY_PATH}")
print(
    "Next: 35_early_payment_default_modeling.ipynb -- builds the restricted-"
    "window feature set (first K statements per customer, same transformations "
    "as Notebook 04) and trains/evaluates the early-detection model against "
    "the KPI target set in Section 6 above."
)
